In [1]:
from inference_functions import load_bit_phoneme_model, evaluate_model, decode_outputs
from dataset import getDatasetLoaders
import numpy as np

/home/ubuntu/miniconda/envs/speech-bci/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_file = '/data2/neural_data/ptDecoder_ctc_both_char_phoneme'
trainLoaders, testLoaders, loadedData = getDatasetLoaders(
        data_file, 8, None, 
        False
    )

In [3]:
device = 'cuda'
bit_phoneme_filepath = "/data2/models/time_masked_transfomer_characters_phonemes_80ms_seed_0/"
model, args = load_bit_phoneme_model(bit_phoneme_filepath)
model = model.to(device)

/home/ubuntu/miniconda/envs/speech-bci/lib/python3.9/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [4]:
outputs, cer, per_day_cer, cer2, per_day_cer2 = evaluate_model(model, loadedData, args, partition='train', device='cuda')


/home/ubuntu/transformers_with_dietcorp/src/neural_decoder/augmentations.py:170: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ../aten/src/ATen/native/Convolution.cpp:895.)
  return self.conv(input, weight=self.weight, groups=self.groups, padding="same")


CER DAY 0: 0.186826
CER2 DAY 0: 0.162962
CER DAY 1: 0.161901
CER2 DAY 1: 0.133007
CER DAY 2: 0.074652
CER2 DAY 2: 0.050287
CER DAY 3: 0.091851
CER2 DAY 3: 0.060693
CER DAY 4: 0.031945
CER2 DAY 4: 0.017411
CER DAY 5: 0.034029
CER2 DAY 5: 0.018944
CER DAY 6: 0.032451
CER2 DAY 6: 0.016165
CER DAY 7: 0.035898
CER2 DAY 7: 0.019511
CER DAY 8: 0.038870
CER2 DAY 8: 0.020582
CER DAY 9: 0.046151
CER2 DAY 9: 0.026858
CER DAY 10: 0.033069
CER2 DAY 10: 0.017935
CER DAY 11: 0.040826
CER2 DAY 11: 0.021844
CER DAY 12: 0.045087
CER2 DAY 12: 0.025732
CER DAY 13: 0.032279
CER2 DAY 13: 0.016071
CER DAY 14: 0.028467
CER2 DAY 14: 0.015460
CER DAY 15: 0.028094
CER2 DAY 15: 0.015482
CER DAY 16: 0.026222
CER2 DAY 16: 0.013389
CER DAY 17: 0.028919
CER2 DAY 17: 0.013060
CER DAY 18: 0.031243
CER2 DAY 18: 0.017085
CER DAY 19: 0.030059
CER2 DAY 19: 0.017168
CER DAY 20: 0.030888
CER2 DAY 20: 0.014920
CER DAY 21: 0.028268
CER2 DAY 21: 0.015239
CER DAY 22: 0.033784
CER2 DAY 22: 0.014962
CER DAY 23: 0.036157
CER2 DAY 2

In [6]:
phoneme_decoded_strs, character_decoded_strs, true_seq_strs = decode_outputs(outputs, 8800)

In [49]:
len(phoneme_decoded_strs)

8800

In [43]:
def topk_labels(logits: np.ndarray, K: int, vocab: list, apply_ctc_rule: bool) -> np.ndarray:
    """
    Args:
        logits: np.ndarray of shape (T, N) where
                N = 1 + len(vocab) (index 0 = CTC blank, rest follow vocab order)
        K: number of top tokens to return
        vocab: list of labels (phonemes or characters)
        apply_ctc_rule: removes blanks and repeats not separated by a blank

    Returns:
        np.ndarray of shape (T, K) with label strings
    """
    # Build id -> label mapping
    
    blank_token = "~"
    
    id2label = [blank_token] + vocab

    # Get top-K indices per timestep
    topk_ids = np.argsort(logits, axis=1)[:, -K:][:, ::-1]

    # Map to labels
    topk_labels = np.vectorize(lambda i: id2label[i])(topk_ids)
    
    top1_ids = np.argsort(logits, axis=1)[:, -1:][:, ::-1]
    
    blank_indices = np.argwhere(top1_ids == 0).squeeze()
    
    # returns indices where the next index is the same
    repeating_indices = np.argwhere(top1_ids[:-1] == top1_ids[1:]).squeeze()
    
    indices_to_remove = np.union1d(blank_indices, repeating_indices+1).squeeze()
    
    indices_to_keep = np.setdiff1d(np.arange(len(top1_ids)), indices_to_remove)

    return topk_labels[indices_to_keep], indices_to_remove.shape[0], indices_to_keep.shape[0], indices_to_keep


In [50]:
# Phone definitions and mappings
PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]
PHONE_DEF_SIL = PHONE_DEF + ["<>"]

CHAR_VOCAB = [
    "<>",          # space token
    "!", ",", ".", "?", "'",   # punctuation (incl. apostrophe)
] + [chr(i) for i in range(ord('a'), ord('z') + 1)]  # 'a'..'z'


character_outputs = []
char_frames_removed = 0
char_frames_kept = 0
char_indices_kept = []

for character_output in outputs['logits']:
    
    topk_chars, cfr, cfk, char_idxs_kept = topk_labels(character_output, K=10, vocab=CHAR_VOCAB, apply_ctc_rule=True)
    character_outputs.append(topk_chars)
    char_frames_removed += cfr
    char_frames_kept += cfk
    char_indices_kept.append(char_idxs_kept)
    
    
phoneme_outputs = []
phoneme_frames_removed = 0
phoneme_frames_kept = 0
phoneme_indices_kept = []

for phoneme_output in outputs['logits2']:
    
    topk_phones, pfr, pfk, _ = topk_labels(phoneme_output, K=10, vocab=PHONE_DEF_SIL, apply_ctc_rule=True)
    phoneme_outputs.append(topk_phones)
    phoneme_frames_removed += pfr
    phoneme_frames_kept += pfk    

In [51]:
print(char_frames_removed, phoneme_frames_removed, char_frames_kept, phoneme_frames_kept)

412072 450508 283421 244985


In [52]:
from typing import Any, Iterable, Sequence

def rows_to_text_dual(
    arr1: Any,
    arr2: Any,
    elem_sep: str = " ",
    row_sep: str = "\n",
    side_sep: str = " | "
) -> str:
    """
    Convert two 2-D arrays (or list-of-lists) to a single string with side-by-side rows.
    - Each element in a row is joined with `elem_sep`.
    - Each pair of rows is joined with `side_sep`.
    - Each line is joined with `row_sep` (newline by default).
    """
    # Support numpy arrays or list-of-lists
    rows1 = arr1.tolist() if isinstance(arr1, np.ndarray) else arr1
    rows2 = arr2.tolist() if isinstance(arr2, np.ndarray) else arr2
    
    if len(rows1) != len(rows2):
        raise ValueError("Both arrays must have the same number of rows")
    
    lines = []
    for r1, r2 in zip(rows1, rows2):
        if isinstance(r1, np.ndarray): r1 = r1.tolist()
        if isinstance(r2, np.ndarray): r2 = r2.tolist()
        left = elem_sep.join(map(str, r1))
        right = elem_sep.join(map(str, r2))
        lines.append(left + side_sep + right)
    
    return row_sep.join(lines)


def rows_to_text(arr: Any, elem_sep: str = " ", row_sep: str = "\n") -> str:
    """
    Convert a 2-D numpy array (or list of lists) to a single string.
    - Each element in a row is joined with `elem_sep`.
    - Each row is joined with `row_sep` (newline by default).
    """
    # Support ndarray or list-of-lists
    rows: Iterable = arr.tolist() if isinstance(arr, np.ndarray) else arr
    
    lines = []
    for r in rows:
        if isinstance(r, np.ndarray):
            r = r.tolist()
        lines.append(elem_sep.join(map(str, r)))
    
    return row_sep.join(lines)

In [55]:
import json
from pathlib import Path
OUT_JSONL = "/data2/jsonl/train.jsonl"
USER_HDR = "<|start_header_id|>user<|end_header_id|>"
ASST_HDR = "<|start_header_id|>assistant<|end_header_id|>"
EOT      = "<|eot_id|>"   # include if your format expects it
prompt = (
    'You are helping decode speech from neural activity to help restore communication for a paralyzed patient. '
    'For each 80 ms time bin of neural activity, a model trained with CTC loss provides the 10 most probable ARPAbet phonemes, listed from most probable to least probable. '
    'The special token "~" denotes the CTC blank and should never be included in the response, and "<>" denotes a space and should be converted to a single space. '
    'The decoded outputs are listed on separate lines ordered by time. '
    'CTC post-processing has already removed outputs where the CTC blank was most probable as well as outputs with consecutive repeats not separated by a CTC blank. ' 
    'Produce fluent, standard spelling and grammar. '
    'Output only the final sentence text, with no explanations or metadata.'
)
eval_mode = False

with Path(OUT_JSONL).open("w", encoding="utf-8") as fout:
    
    for idx in range(len(phoneme_outputs)):
        
        topk_text= rows_to_text(phoneme_outputs[idx])
        
        if eval_mode:
            ground_truth = ""
        else:
            ground_truth = true_seq_strs[idx]
        
        msg = (
            f"{USER_HDR}\n\n"
            f"{prompt}\n\n"
            f"{topk_text}\n\n"
            f"{ASST_HDR}\n\n"
            f"{ground_truth}{EOT}"
        )
        
        obj = {"text": msg}
        
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [12]:
with open("/data2/jsonl/val_ground_truth.json", "w") as f:
    json.dump(true_seq_strs, f)

['theocracy reconsidered',
 'rich purchased several signed lithographs',
 'so rules we made in unabashed collusion',
 "lori's costume needed black gloves to be completely elegant",
 "the tooth fairy forgot to come when roger's tooth fell out",
 'that stinging vapor was caused by chloride vaporization',
 "before thursday's exam review every formula",
 'wildfire near sunshine forces park closures',
 "the word means it won't boil away easily nothing else",
 "would a blue feather in a man's hat make him happy all day",
 'he talked about unauthentic storylines too',
 'most young rise early every morning',
 'with this no loyal citizen can quarrel',
 'primitive tribes have an upbeat attitude',
 'and it was not a judge but a justice of the peace who made the decision',
 'we seemed to be witnessing the population explosion right in our own backyards',
 'sometimes desperate measures can also turn into solid moves for the future',
 'those answers will be straightforward if you think them through 